# Qwen3.8-27B on Kaggle (2x T4) with llama-server + ngrok

OpenAI-compatible endpoint for a coding harness (tool calling, streaming, reasoning field).

**Before running**
1. Notebook settings: **Accelerator = GPU T4 x2**, **Internet = On**.
2. Add-ons -> Secrets (enable each for this notebook):
   - `NGROK_AUTHTOKEN` (required)
   - `NGROK_DOMAIN` (optional, your static ngrok domain, e.g. `something.ngrok-free.app`)
   - `API_KEY` (recommended, a long random string; if missing, one is generated and printed)
   - `HF_TOKEN` (optional, only if downloads get rate limited)
3. Run all cells in order. The first run builds llama.cpp (about 8-15 min) and downloads the model (about 20 GB).
4. The last cell keeps the session alive. Leave it running while you work.

**Harness settings** are printed after the ngrok cell: base URL (`.../v1`), API key and model name.

In [ ]:
import os, re, sys, json, time, shutil, secrets, subprocess

# ======================= CONFIG (edit here) =======================
MODEL_REPO    = "unsloth/Qwen3.8-27B-GGUF"
QUANT         = "Q5_K_M"            # ~19.8 GB. For more speed and headroom try "Q4_K_M" (~16.5 GB)
CTX_SIZE      = 92160               # ~90K tokens (single slot)
CTX_FALLBACKS = [65536, 49152]      # tried automatically if loading fails (usually out of VRAM)
KV_TYPE       = "q8_0"              # KV-cache precision; q8_0 halves memory vs f16
ALIAS         = "qwen3.8-27b"       # model name to use in your harness
PORT          = 8080
THINKING      = False               # server default; clients can override per request
WORKDIR       = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
# ===================================================================


def get_secret(name, default=None):
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(name)
        if val:
            return val.strip()
    except Exception:
        pass
    return os.environ.get(name, default)


def sh(cmd, cwd=None, check=True, quiet=False):
    shell = isinstance(cmd, str)
    if not quiet:
        print("$", cmd if shell else " ".join(cmd))
    p = subprocess.Popen(cmd, shell=shell, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if not quiet:
            print(line, end="")
    p.wait()
    if p.returncode != 0 and check:
        if quiet:
            print("".join(lines[-60:]))
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {cmd}")
    return p.returncode


def gpu_mem():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True).stdout.strip().splitlines()
        return " | ".join(f"GPU{i}: {l.strip()}" for i, l in enumerate(out))
    except Exception:
        return "n/a"


NGROK_TOKEN = get_secret("NGROK_AUTHTOKEN")
if not NGROK_TOKEN:
    raise RuntimeError("Add NGROK_AUTHTOKEN to Kaggle Secrets (Add-ons -> Secrets) and enable it for this notebook.")

NGROK_DOMAIN = (get_secret("NGROK_DOMAIN") or "").replace("https://", "").replace("http://", "").strip().strip("/").split("/")[0]
HF_TOKEN = get_secret("HF_TOKEN")

API_KEY = get_secret("API_KEY")
KEY_GENERATED = False
if not API_KEY:
    API_KEY = "sk-" + secrets.token_urlsafe(32)
    KEY_GENERATED = True
    print("API_KEY secret not found -> generated a temporary key (shown below). Save it as a secret to keep it stable.")

try:
    gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
except Exception:
    gpus = ""
print(gpus or "No GPU detected!")
if len(gpus.splitlines()) < 2:
    print("WARNING: fewer than 2 GPUs visible. Set the accelerator to 'GPU T4 x2' and restart the session.")
print("Work dir:", WORKDIR)

In [ ]:
sh("pip install -q -U pyngrok huggingface_hub requests", quiet=True)
sh("pip install -q hf_transfer", check=False, quiet=True)
try:
    import hf_transfer  # noqa: F401
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
except ImportError:
    pass
print("Dependencies ready.")

In [ ]:
# Build llama.cpp (CUDA, T4 = compute capability 7.5). Skipped if already built in this session.
LLAMA_DIR = f"{WORKDIR}/llama.cpp"
SERVER_BIN = f"{LLAMA_DIR}/build/bin/llama-server"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")

if os.path.exists(SERVER_BIN):
    print("llama-server already built:", SERVER_BIN)
else:
    if shutil.which("nvcc") is None:
        raise RuntimeError("nvcc (CUDA compiler) not found. Make sure a GPU accelerator is selected.")
    if shutil.which("cmake") is None:
        sh("pip install -q cmake", quiet=True)
    if os.path.isdir(LLAMA_DIR):
        shutil.rmtree(LLAMA_DIR)
    print("Cloning llama.cpp ...")
    sh(f"git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_DIR}", quiet=True)
    print("Configuring ...")
    sh("cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF",
       cwd=LLAMA_DIR, quiet=True)
    print("Building llama-server (this takes several minutes) ...")
    t0 = time.time()
    sh(f"cmake --build build --config Release -j {os.cpu_count()} --target llama-server", cwd=LLAMA_DIR, quiet=True)
    print(f"Build finished in {(time.time() - t0) / 60:.1f} min")

if not os.path.exists(SERVER_BIN):
    raise RuntimeError("llama-server binary not found after build.")
sh([SERVER_BIN, "--version"], check=False)

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download

MODEL_DIR = f"{WORKDIR}/models"
os.makedirs(MODEL_DIR, exist_ok=True)

all_files = list_repo_files(MODEL_REPO, token=HF_TOKEN)
ggufs = [f for f in all_files if f.lower().endswith(".gguf") and "mmproj" not in f.lower()]
files = [f for f in ggufs if QUANT.lower() in f.lower()]
if not files:
    raise RuntimeError(f"No GGUF matching {QUANT}. Available: " + ", ".join(sorted(ggufs)))

ud = [f for f in files if "ud-" in f.lower()]
files = sorted(ud or files)


def shard_base(f):
    return re.sub(r"-\d{5}-of-\d{5}\.gguf$", ".gguf", f)


chosen = sorted(f for f in files if shard_base(f) == shard_base(files[0]))
print("Downloading:", chosen)
paths = [hf_hub_download(MODEL_REPO, f, local_dir=MODEL_DIR, token=HF_TOKEN) for f in chosen]
MODEL_PATH = paths[0]
total_gb = sum(os.path.getsize(p) for p in paths) / 1e9
print(f"Model ready: {MODEL_PATH} ({total_gb:.1f} GB)")

In [ ]:
import requests

LOG_PATH = f"{WORKDIR}/llama-server.log"
server_proc = globals().get("server_proc")


def stop_server():
    global server_proc
    if server_proc is not None and server_proc.poll() is None:
        server_proc.terminate()
        try:
            server_proc.wait(20)
        except subprocess.TimeoutExpired:
            server_proc.kill()
    server_proc = None


def tail_log(n=40):
    try:
        with open(LOG_PATH, errors="ignore") as fh:
            return "".join(fh.readlines()[-n:])
    except Exception:
        return ""


def build_cmd(ctx):
    return [
        SERVER_BIN, "-m", MODEL_PATH, "--alias", ALIAS,
        "--host", "127.0.0.1", "--port", str(PORT),
        "-c", str(ctx), "-np", "1",                  # one slot gets the whole context
        "-ngl", "99", "-sm", "layer", "-ts", "1,1",  # all layers on GPU, split across both T4s
        "-fa", "on", "-ctk", KV_TYPE, "-ctv", KV_TYPE,
        "-b", "2048", "-ub", "512",
        "--jinja",                                   # proper chat template -> tool calling works
        "--reasoning-format", "deepseek",            # thinking returned in reasoning_content
        "--no-context-shift",                        # error on overflow instead of silently dropping the system prompt
        "--cache-reuse", "256",
        "--temp", "0.6", "--top-p", "0.95", "--top-k", "20", "--min-p", "0.0",
        "--chat-template-kwargs", json.dumps({"enable_thinking": THINKING}),
        "--api-key", API_KEY,
    ]


def wait_ready(proc, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            return False
        try:
            r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3)
            if r.status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(3)
    return False


stop_server()
ACTIVE_CTX = None
for ctx in [CTX_SIZE] + CTX_FALLBACKS:
    cmd = build_cmd(ctx)
    shown = " ".join("***" if i > 0 and cmd[i - 1] == "--api-key" else c for i, c in enumerate(cmd))
    print(f"\nStarting llama-server with context {ctx} ...\n{shown}\n")
    log = open(LOG_PATH, "w")
    server_proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    if wait_ready(server_proc):
        ACTIVE_CTX = ctx
        break
    print("Server did not become ready. Last log lines:\n" + tail_log(30))
    stop_server()

if ACTIVE_CTX is None:
    raise RuntimeError("llama-server could not start. Try QUANT = 'Q4_K_M' or a smaller CTX_SIZE.")

print(f"llama-server is up (context {ACTIVE_CTX}).")
print(gpu_mem())

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)
try:
    ngrok.kill()
except Exception:
    pass

if NGROK_DOMAIN:
    try:
        tunnel = ngrok.connect(addr=PORT, proto="http", domain=NGROK_DOMAIN)
    except Exception as e:
        print("domain= failed, retrying with url= :", repr(e))
        tunnel = ngrok.connect(addr=PORT, proto="http", url="https://" + NGROK_DOMAIN)
else:
    tunnel = ngrok.connect(addr=PORT, proto="http")

PUBLIC_URL = tunnel.public_url.replace("http://", "https://")
key_shown = API_KEY if KEY_GENERATED else API_KEY[:4] + "..." + "  (your API_KEY secret)"

print("=" * 66)
print("SERVER ONLINE")
print("=" * 66)
print(f"Base URL   : {PUBLIC_URL}/v1")
print(f"Model name : {ALIAS}")
print(f"API key    : {key_shown}")
print(f"Context    : {ACTIVE_CTX} tokens")
print("-" * 66)
print("Generic OpenAI-compatible settings:")
print(f"  OPENAI_BASE_URL={PUBLIC_URL}/v1")
print("  OPENAI_API_KEY=<your key>")
print(f"  model={ALIAS}")
print("Endpoints: /v1/chat/completions  /v1/completions  /v1/models  /health")
print("If your client hits an ngrok warning page, add header: ngrok-skip-browser-warning: 1")
print("=" * 66)

In [ ]:
# Smoke tests through the public tunnel: models, chat + speed, streaming, tool calling.
BASE = PUBLIC_URL + "/v1"
HDR = {"Authorization": f"Bearer {API_KEY}", "ngrok-skip-browser-warning": "1", "Content-Type": "application/json"}


def chat(payload, stream=False, timeout=600):
    return requests.post(BASE + "/chat/completions", headers=HDR, json=payload, timeout=timeout, stream=stream)


r = requests.get(BASE + "/models", headers=HDR, timeout=30)
print("models:", r.status_code, [m.get("id") for m in r.json().get("data", [])] if r.ok else r.text[:200])

r = chat({"model": ALIAS, "max_tokens": 160, "temperature": 0.2,
          "messages": [{"role": "user", "content": "Write a one-line C++ function that returns the square of an int."}]})
if r.ok:
    j = r.json()
    print("\nchat reply:", j["choices"][0]["message"].get("content", "").strip())
    t = j.get("timings", {})
    print(f"speed -> prompt: {t.get('prompt_per_second', 0):.1f} tok/s | generation: {t.get('predicted_per_second', 0):.1f} tok/s")
else:
    print("chat failed:", r.status_code, r.text[:300])

r = chat({"model": ALIAS, "max_tokens": 48, "stream": True,
          "messages": [{"role": "user", "content": "Count from 1 to 10."}]}, stream=True)
chunks, done = 0, False
for line in r.iter_lines():
    if line and line.startswith(b"data:"):
        if b"[DONE]" in line:
            done = True
            break
        chunks += 1
print(f"\nstreaming: {'OK' if chunks > 0 and done else 'FAILED'} ({chunks} chunks)")

tools = [{"type": "function", "function": {
    "name": "read_file", "description": "Read a file from the project",
    "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}}}]
r = chat({"model": ALIAS, "max_tokens": 256, "temperature": 0, "tools": tools, "tool_choice": "auto",
          "messages": [{"role": "user", "content": "Open Source/MyGame/Public/MyActor.h and tell me what it declares."}]})
if r.ok:
    msg = r.json()["choices"][0]["message"]
    calls = msg.get("tool_calls") or []
    print("tool calling:", "OK -> " + json.dumps(calls[0]["function"]) if calls else "no tool call returned: " + str(msg.get("content"))[:200])
else:
    print("tool test failed:", r.status_code, r.text[:300])

In [ ]:
# Keep the session alive and watch the server. Stop this cell (interrupt) to shut everything down.
print("Server running. Leave this cell running while you use the harness.")
print(f"Base URL: {PUBLIC_URL}/v1  | model: {ALIAS}")
try:
    n = 0
    while True:
        time.sleep(60)
        if server_proc is None or server_proc.poll() is not None:
            print("llama-server exited. Last log lines:\n" + tail_log(40))
            break
        n += 1
        if n % 5 == 0:
            print(time.strftime("%H:%M:%S"), "server OK |", gpu_mem())
except KeyboardInterrupt:
    print("Interrupted.")
finally:
    stop_server()
    try:
        ngrok.kill()
    except Exception:
        pass
    print("Server and tunnel stopped.")